<a href="https://colab.research.google.com/github/ikabrain/UCS772-CV-NLP-Lab/blob/main/NLP_assign2/NLP_assign2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Assignment 2 - NLP pre-processing
---

## Introduction
---

This notebook goes through a necessary step of any data science project - data cleaning. Data cleaning is a time consuming and unenjoyable task, yet it's a very important one. Keep in mind, "garbage in, garbage out". Feeding dirty data into a model will give us results that are meaningless.

Objectives:

 - **Getting the data:** In this case, we'll be scraping data from a website.
 - **Cleaning the data:** We will walk through popular text pre-processing techniques.
 - **Organizing the data:** We will organize the cleaned data into a way that is easy to input into other algorithms.

The output of this notebook will be clean, organized data in two standard text formats:

1. **Corpus:** A collection of text
2. **Document-Term Matrix:** Word counts in matrix format

## Problem Statement
---

Look at transcripts (similar to transcripts of some comedians from [Scraps From The Loft](https://scrapsfromtheloft.com)), pre-
process and organize the data to note their similarities and differences to find various patterns.

### Getting The Data

Luckily, there are wonderful people online that keep track of stand up routine transcripts. [Scraps From The Loft](https://scrapsfromtheloft.com) makes them available for non-profit and educational purposes.

To decide which comedians to look into, I went on IMDB and looked specifically at comedy specials that were released in the past 5 years. To narrow it down further, I looked only at those with greater than a 7.5/10 rating and more than 2000 votes. If a comedian had multiple specials that fit those requirements, I would pick the most highly rated one. I ended up with a dozen comedy specials.

In [1]:
# Web scraping, pickle imports
import requests
from bs4 import BeautifulSoup
import pickle

# Scrapes transcript data from scrapsfromtheloft.com
def url_to_transcript(url):
    '''Returns transcript data specifically from scrapsfromtheloft.com.'''
    page = requests.get(url).text
    soup = BeautifulSoup(page, "lxml")
    #text = [p.text for p in soup.find(class_="ast-container").find_all('p')]
    # page html code was modified this line of code can be formated like this
    text = [p.text for p in soup.find_all('p')]
    print(url)
    return text

# URLs of transcripts in scope
urls = [
    'https://scrapsfromtheloft.com/2017/05/06/louis-ck-oh-my-god-full-transcript/',
    'https://scrapsfromtheloft.com/2017/04/11/dave-chappelle-age-spin-2017-full-transcript/',
    'https://scrapsfromtheloft.com/2018/03/15/ricky-gervais-humanity-transcript/',
    'https://scrapsfromtheloft.com/comedy/bo-burnham-what-transcript/',
    'https://scrapsfromtheloft.com/2017/05/24/bill-burr-im-sorry-feel-way-2014-full-transcript/',
    'https://scrapsfromtheloft.com/2017/04/21/jim-jefferies-bare-2014-full-transcript/',
    'https://scrapsfromtheloft.com/2017/08/02/john-mulaney-comeback-kid-2015-full-transcript/',
    'https://scrapsfromtheloft.com/comedy/hasan-minhaj-homecoming-king-transcript/',
    'https://scrapsfromtheloft.com/2017/09/19/ali-wong-baby-cobra-2016-full-transcript/',
    'https://scrapsfromtheloft.com/2017/08/03/anthony-jeselnik-thoughts-prayers-2015-full-transcript/',
    'https://scrapsfromtheloft.com/2018/03/03/mike-birbiglia-my-girlfriends-boyfriend-2013-full-transcript/',
    'https://scrapsfromtheloft.com/2017/08/19/joe-rogan-triggered-2016-full-transcript/',
]

# Comedian names
comedians = ['louis', 'dave', 'ricky', 'bo', 'bill', 'jim', 'john', 'hasan', 'ali', 'anthony', 'mike', 'joe']

In [2]:
# Actually request transcripts (takes a few minutes to run)
transcripts = [url_to_transcript(u) for u in urls]

https://scrapsfromtheloft.com/2017/05/06/louis-ck-oh-my-god-full-transcript/
https://scrapsfromtheloft.com/2017/04/11/dave-chappelle-age-spin-2017-full-transcript/
https://scrapsfromtheloft.com/2018/03/15/ricky-gervais-humanity-transcript/
https://scrapsfromtheloft.com/comedy/bo-burnham-what-transcript/
https://scrapsfromtheloft.com/2017/05/24/bill-burr-im-sorry-feel-way-2014-full-transcript/
https://scrapsfromtheloft.com/2017/04/21/jim-jefferies-bare-2014-full-transcript/
https://scrapsfromtheloft.com/2017/08/02/john-mulaney-comeback-kid-2015-full-transcript/
https://scrapsfromtheloft.com/comedy/hasan-minhaj-homecoming-king-transcript/
https://scrapsfromtheloft.com/2017/09/19/ali-wong-baby-cobra-2016-full-transcript/
https://scrapsfromtheloft.com/2017/08/03/anthony-jeselnik-thoughts-prayers-2015-full-transcript/
https://scrapsfromtheloft.com/2018/03/03/mike-birbiglia-my-girlfriends-boyfriend-2013-full-transcript/
https://scrapsfromtheloft.com/2017/08/19/joe-rogan-triggered-2016-full-t

In [3]:
# Pickle files for later use

# Make a new directory to hold the text files
!rm -rf transcripts && mkdir -p transcripts

for i, c in enumerate(comedians):
    with open("transcripts/" + c + ".txt", "wb") as file:
        pickle.dump(transcripts[i], file)

In [4]:
# Load pickled files
data = {}
for i, c in enumerate(comedians):
    with open("transcripts/" + c + ".txt", "rb") as file:
        data[c] = pickle.load(file)

In [5]:
# Double check to make sure data has been loaded properly
data.keys()

dict_keys(['louis', 'dave', 'ricky', 'bo', 'bill', 'jim', 'john', 'hasan', 'ali', 'anthony', 'mike', 'joe'])

In [6]:
# More checks
data['louis'][:2]

['Magazine of culture and entertainment',
 '‘Oh My God’ is the fifth comedy special performed by Louis C.K.. It premiered on HBO on April 13, 2013. Filmed in Phoenix, Arizona at the Celebrity Theatre']

In [7]:
# Let's take a look at our data again
next(iter(data.keys()))

'louis'

In [8]:
# Notice that our dictionary is currently in key: comedian, value: list of text format
next(iter(data.values()))

['Magazine of culture and entertainment',
 '‘Oh My God’ is the fifth comedy special performed by Louis C.K.. It premiered on HBO on April 13, 2013. Filmed in Phoenix, Arizona at the Celebrity Theatre',
 'Intro\nFade the music out. Let’s roll. Hold there. Lights. Do the lights. Thank you. Thank you very much. I appreciate that. I don’t necessarily agree with you, but I appreciate very much. Well, this is a nice place. This is easily the nicest place For many miles in every direction. That’s how you compliment a building And shit on a town with one sentence. It is odd around here, as I was driving here. There doesn’t seem to be any difference Between the sidewalk and the street for pedestrians here. People just kind of walk in the middle of the road. I love traveling And seeing all the different parts of the country. I live in New York. I live in a– There’s no value to your doing that at all.',
 '“The Old Lady And The Dog”\nI live– I live in New York. I always– Like, there’s this old lad

In [9]:
# We are going to change this to key: comedian, value: string format
def combine_text(list_of_text):
    '''Takes a list of text and combines them into one large chunk of text.'''
    combined_text = ' '.join(list_of_text)
    return combined_text

In [10]:
# Combine it!
data_combined = {key: [combine_text(value)] for (key, value) in data.items()}

In [11]:
# We can either keep it in dictionary format or put it into a pandas dataframe
import pandas as pd
pd.set_option('max_colwidth',150)

data_df = pd.DataFrame.from_dict(data_combined).transpose()
data_df.columns = ['transcript']
data_df = data_df.sort_index()
data_df

,transcript
ali,"Magazine of culture and entertainment Ali Wong’s stand up special delves into her sexual adventures, hoarding, the rocky road to pregnancy, and wh..."
anthony,Magazine of culture and entertainment There’s no subject too dark as the comedian skewers taboos and riffs on national tragedies before pulling ba...
bill,"Magazine of culture and entertainment [cheers and applause] All right, thank you! Thank you very much! Thank you. Thank you. Thank you. How are yo..."
bo,Magazine of culture and entertainment Bo What? Old MacDonald had a farm E I E I O And on that farm he had a pig E I E I O Here a snort There a Old...
dave,Magazine of culture and entertainment The Age of Spin: Dave Chappelle Live at the Hollywood Palladium (2017) Dave Chappelle gives his usual skewed...
hasan,"Magazine of culture and entertainment Comic Hasan Minhaj of “The Daily Show” shares personal stories about racism, immigrant parents, prom night h..."
jim,"Magazine of culture and entertainment Nothing is sacred in this show from Australian comic Jim Jefferies, whether it’s the mother of his child, au..."
joe,
john,"Magazine of culture and entertainment Armed with boyish charm and a sharp wit, the former “SNL” writer John Mulaney offers sly takes on marriage, ..."
louis,"Magazine of culture and entertainment ‘Oh My God’ is the fifth comedy special performed by Louis C.K.. It premiered on HBO on April 13, 2013. Film..."


## Q1: Cleaning The Data
---

When dealing with numerical data, data cleaning often involves removing null values and duplicate data, dealing with outliers, etc. With text data, there are some common data cleaning techniques, which are also known as text pre-processing techniques.

With text data, this cleaning process can go on forever. There's always an exception to every cleaning step. So, we're going to follow the MVP (minimum viable product) approach - start simple and iterate. Here are a bunch of things you can do to clean your data. We're going to execute just the common cleaning steps here and the rest can be done at a later point to improve our results.

**Common data cleaning steps on all text:**
* Make text all lower case
* Remove punctuation
* Remove numerical values
* Remove common non-sensical text (/n)
* Tokenize text
* Remove stop words

**More data cleaning steps after tokenization:**
* Stemming / lemmatization
* Parts of speech tagging
* Create bi-grams or tri-grams
* Deal with typos
* And more...

In [12]:
# Let's take a look at the transcript for Ali Wong
data_df.transcript.loc['ali']

'Magazine of culture and entertainment Ali Wong’s stand up special delves into her sexual adventures, hoarding, the rocky road to pregnancy, and why feminism is terrible Ladies and gentlemen, please welcome to the stage: Ali Wong! Hi. Hello! Welcome! Thank you! Thank you for coming. Hello! Hello. We are gonna have to get this shit over with, ’cause I have to pee in, like, ten minutes. But thank you, everybody, so much for coming. Um… It’s a very exciting day for me. It’s been a very exciting year for me. I turned 33 this year. Yes! Thank you, five people. I appreciate that. Uh, I can tell that I’m getting older, because, now, when I see an 18-year-old girl, my automatic thought… is “Fuck you.” “Fuck you. I don’t even know you, but fuck you!” ‘Cause I’m straight up jealous. I’m jealous, first and foremost, of their metabolism. Because 18-year-old girls, they could just eat like shit, and then they take a shit and have a six-pack, right? They got that-that beautiful inner thigh clearance

Perform the following data cleaning on transcripts:
 1. Make text all lower case
 2. Remove punctuation
 3. Remove numerical values
 4. Remove common non-sensical text (\n)
 5. Tokenize text
 6. Remove stop words

In [13]:
# Apply a first round of text cleaning techniques
import re
import string

def clean_text_round1(text):
    '''Make text lowercase, remove text in square brackets, remove punctuation and remove words containing numbers.'''
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    return text

round1 = lambda x: clean_text_round1(x)

In [14]:
# Let's take a look at the updated text
data_clean = pd.DataFrame(data_df.transcript.apply(round1))
data_clean

,transcript
ali,magazine of culture and entertainment ali wong’s stand up special delves into her sexual adventures hoarding the rocky road to pregnancy and why f...
anthony,magazine of culture and entertainment there’s no subject too dark as the comedian skewers taboos and riffs on national tragedies before pulling ba...
bill,magazine of culture and entertainment all right thank you thank you very much thank you thank you thank you how are you what’s going on thank you...
bo,magazine of culture and entertainment bo what old macdonald had a farm e i e i o and on that farm he had a pig e i e i o here a snort there a old ...
dave,magazine of culture and entertainment the age of spin dave chappelle live at the hollywood palladium dave chappelle gives his usual skewed insigh...
hasan,magazine of culture and entertainment comic hasan minhaj of “the daily show” shares personal stories about racism immigrant parents prom night hor...
jim,magazine of culture and entertainment nothing is sacred in this show from australian comic jim jefferies whether it’s the mother of his child audi...
joe,
john,magazine of culture and entertainment armed with boyish charm and a sharp wit the former “snl” writer john mulaney offers sly takes on marriage hi...
louis,magazine of culture and entertainment ‘oh my god’ is the fifth comedy special performed by louis ck it premiered on hbo on april filmed in phoen...


In [15]:
# Apply a second round of cleaning
def clean_text_round2(text):
    '''Get rid of some additional punctuation and non-sensical text that was missed the first time around.'''
    text = re.sub('[‘’“”…]', '', text)
    text = re.sub('\n', '', text)
    return text

round2 = lambda x: clean_text_round2(x)

In [16]:
# Let's take a look at the updated text
data_clean = pd.DataFrame(data_clean.transcript.apply(round2))
data_clean

,transcript
ali,magazine of culture and entertainment ali wongs stand up special delves into her sexual adventures hoarding the rocky road to pregnancy and why fe...
anthony,magazine of culture and entertainment theres no subject too dark as the comedian skewers taboos and riffs on national tragedies before pulling bac...
bill,magazine of culture and entertainment all right thank you thank you very much thank you thank you thank you how are you whats going on thank you ...
bo,magazine of culture and entertainment bo what old macdonald had a farm e i e i o and on that farm he had a pig e i e i o here a snort there a old ...
dave,magazine of culture and entertainment the age of spin dave chappelle live at the hollywood palladium dave chappelle gives his usual skewed insigh...
hasan,magazine of culture and entertainment comic hasan minhaj of the daily show shares personal stories about racism immigrant parents prom night horro...
jim,magazine of culture and entertainment nothing is sacred in this show from australian comic jim jefferies whether its the mother of his child audit...
joe,
john,magazine of culture and entertainment armed with boyish charm and a sharp wit the former snl writer john mulaney offers sly takes on marriage his ...
louis,magazine of culture and entertainment oh my god is the fifth comedy special performed by louis ck it premiered on hbo on april filmed in phoenix...


In [17]:
# v) Tokenize text - split each cleaned transcript into individual word tokens
data_clean['tokens'] = data_clean.transcript.str.split()
data_clean

,transcript,tokens
ali,magazine of culture and entertainment ali wongs stand up special delves into her sexual adventures hoarding the rocky road to pregnancy and why fe...,"[magazine, of, culture, and, entertainment, ali, wongs, stand, up, special, delves, into, her, sexual, adventures, hoarding, the, rocky, road, to,..."
anthony,magazine of culture and entertainment theres no subject too dark as the comedian skewers taboos and riffs on national tragedies before pulling bac...,"[magazine, of, culture, and, entertainment, theres, no, subject, too, dark, as, the, comedian, skewers, taboos, and, riffs, on, national, tragedie..."
bill,magazine of culture and entertainment all right thank you thank you very much thank you thank you thank you how are you whats going on thank you ...,"[magazine, of, culture, and, entertainment, all, right, thank, you, thank, you, very, much, thank, you, thank, you, thank, you, how, are, you, wha..."
bo,magazine of culture and entertainment bo what old macdonald had a farm e i e i o and on that farm he had a pig e i e i o here a snort there a old ...,"[magazine, of, culture, and, entertainment, bo, what, old, macdonald, had, a, farm, e, i, e, i, o, and, on, that, farm, he, had, a, pig, e, i, e, ..."
dave,magazine of culture and entertainment the age of spin dave chappelle live at the hollywood palladium dave chappelle gives his usual skewed insigh...,"[magazine, of, culture, and, entertainment, the, age, of, spin, dave, chappelle, live, at, the, hollywood, palladium, dave, chappelle, gives, his,..."
hasan,magazine of culture and entertainment comic hasan minhaj of the daily show shares personal stories about racism immigrant parents prom night horro...,"[magazine, of, culture, and, entertainment, comic, hasan, minhaj, of, the, daily, show, shares, personal, stories, about, racism, immigrant, paren..."
jim,magazine of culture and entertainment nothing is sacred in this show from australian comic jim jefferies whether its the mother of his child audit...,"[magazine, of, culture, and, entertainment, nothing, is, sacred, in, this, show, from, australian, comic, jim, jefferies, whether, its, the, mothe..."
joe,,[]
john,magazine of culture and entertainment armed with boyish charm and a sharp wit the former snl writer john mulaney offers sly takes on marriage his ...,"[magazine, of, culture, and, entertainment, armed, with, boyish, charm, and, a, sharp, wit, the, former, snl, writer, john, mulaney, offers, sly, ..."
louis,magazine of culture and entertainment oh my god is the fifth comedy special performed by louis ck it premiered on hbo on april filmed in phoenix...,"[magazine, of, culture, and, entertainment, oh, my, god, is, the, fifth, comedy, special, performed, by, louis, ck, it, premiered, on, hbo, on, ap..."


In [18]:
# vi) Remove stop words - filter common English stop words out of the tokens
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

data_clean['tokens_no_stop'] = data_clean.tokens.apply(
    lambda tokens: [t for t in tokens if t not in ENGLISH_STOP_WORDS]
)
data_clean

,transcript,tokens,tokens_no_stop
ali,magazine of culture and entertainment ali wongs stand up special delves into her sexual adventures hoarding the rocky road to pregnancy and why fe...,"[magazine, of, culture, and, entertainment, ali, wongs, stand, up, special, delves, into, her, sexual, adventures, hoarding, the, rocky, road, to,...","[magazine, culture, entertainment, ali, wongs, stand, special, delves, sexual, adventures, hoarding, rocky, road, pregnancy, feminism, terrible, l..."
anthony,magazine of culture and entertainment theres no subject too dark as the comedian skewers taboos and riffs on national tragedies before pulling bac...,"[magazine, of, culture, and, entertainment, theres, no, subject, too, dark, as, the, comedian, skewers, taboos, and, riffs, on, national, tragedie...","[magazine, culture, entertainment, theres, subject, dark, comedian, skewers, taboos, riffs, national, tragedies, pulling, curtain, provocative, st..."
bill,magazine of culture and entertainment all right thank you thank you very much thank you thank you thank you how are you whats going on thank you ...,"[magazine, of, culture, and, entertainment, all, right, thank, you, thank, you, very, much, thank, you, thank, you, thank, you, how, are, you, wha...","[magazine, culture, entertainment, right, thank, thank, thank, thank, thank, whats, going, thank, pleasure, greater, atlanta, georgia, area, oasis..."
bo,magazine of culture and entertainment bo what old macdonald had a farm e i e i o and on that farm he had a pig e i e i o here a snort there a old ...,"[magazine, of, culture, and, entertainment, bo, what, old, macdonald, had, a, farm, e, i, e, i, o, and, on, that, farm, he, had, a, pig, e, i, e, ...","[magazine, culture, entertainment, bo, old, macdonald, farm, e, e, o, farm, pig, e, e, o, snort, old, macdonald, farm, e, e, o, bo, burnham, hes, ..."
dave,magazine of culture and entertainment the age of spin dave chappelle live at the hollywood palladium dave chappelle gives his usual skewed insigh...,"[magazine, of, culture, and, entertainment, the, age, of, spin, dave, chappelle, live, at, the, hollywood, palladium, dave, chappelle, gives, his,...","[magazine, culture, entertainment, age, spin, dave, chappelle, live, hollywood, palladium, dave, chappelle, gives, usual, skewed, insight, topics,..."
hasan,magazine of culture and entertainment comic hasan minhaj of the daily show shares personal stories about racism immigrant parents prom night horro...,"[magazine, of, culture, and, entertainment, comic, hasan, minhaj, of, the, daily, show, shares, personal, stories, about, racism, immigrant, paren...","[magazine, culture, entertainment, comic, hasan, minhaj, daily, shares, personal, stories, racism, immigrant, parents, prom, night, horrors, stand..."
jim,magazine of culture and entertainment nothing is sacred in this show from australian comic jim jefferies whether its the mother of his child audit...,"[magazine, of, culture, and, entertainment, nothing, is, sacred, in, this, show, from, australian, comic, jim, jefferies, whether, its, the, mothe...","[magazine, culture, entertainment, sacred, australian, comic, jim, jefferies, mother, child, auditioning, disabled, actors, gun, control, ladies, ..."
joe,,[],[]
john,magazine of culture and entertainment armed with boyish charm and a sharp wit the former snl writer john mulaney offers sly takes on marriage his ...,"[magazine, of, culture, and, entertainment, armed, with, boyish, charm, and, a, sharp, wit, the, former, snl, writer, john, mulaney, offers, sly, ...","[magazine, culture, entertainment, armed, boyish, charm, sharp, wit, snl, writer, john, mulaney, offers, sly, takes, marriage, beef, babies, time,..."
louis,magazine of culture and entertainment oh my god is the fifth comedy special performed by louis ck it premiered on hbo on april filmed in phoenix...,"[magazine, of, culture, and, entertainment, oh, my, god, is, the, fifth, comedy, special, performed, by, louis, ck, it, premiered,

**NOTE:** This data cleaning aka text pre-processing step could go on for a while, but we are going to stop for now. After going through some analysis techniques, if you see that the results don't make sense or could be improved, you can come back and make more edits such as:
* Mark 'cheering' and 'cheer' as the same word (stemming / lemmatization)
* Combine 'thank you' into one term (bi-grams)
* And a lot more...

## Q2: Organizing The Data
---

Organize the data into 2 standard text formats:

### a) Corpus

We already created a corpus in an earlier step. The definition of a corpus is a collection of texts, and they are all put together neatly in a pandas dataframe here.

In [19]:
# Let's take a look at our dataframe
data_df

,transcript
ali,"Magazine of culture and entertainment Ali Wong’s stand up special delves into her sexual adventures, hoarding, the rocky road to pregnancy, and wh..."
anthony,Magazine of culture and entertainment There’s no subject too dark as the comedian skewers taboos and riffs on national tragedies before pulling ba...
bill,"Magazine of culture and entertainment [cheers and applause] All right, thank you! Thank you very much! Thank you. Thank you. Thank you. How are yo..."
bo,Magazine of culture and entertainment Bo What? Old MacDonald had a farm E I E I O And on that farm he had a pig E I E I O Here a snort There a Old...
dave,Magazine of culture and entertainment The Age of Spin: Dave Chappelle Live at the Hollywood Palladium (2017) Dave Chappelle gives his usual skewed...
hasan,"Magazine of culture and entertainment Comic Hasan Minhaj of “The Daily Show” shares personal stories about racism, immigrant parents, prom night h..."
jim,"Magazine of culture and entertainment Nothing is sacred in this show from Australian comic Jim Jefferies, whether it’s the mother of his child, au..."
joe,
john,"Magazine of culture and entertainment Armed with boyish charm and a sharp wit, the former “SNL” writer John Mulaney offers sly takes on marriage, ..."
louis,"Magazine of culture and entertainment ‘Oh My God’ is the fifth comedy special performed by Louis C.K.. It premiered on HBO on April 13, 2013. Film..."


In [20]:
# Let's add the comedians' full names as well
full_names = ['Ali Wong', 'Anthony Jeselnik', 'Bill Burr', 'Bo Burnham', 'Dave Chappelle', 'Hasan Minhaj',
              'Jim Jefferies', 'Joe Rogan', 'John Mulaney', 'Louis C.K.', 'Mike Birbiglia', 'Ricky Gervais']

data_df['full_name'] = full_names
data_df

,transcript,full_name
ali,"Magazine of culture and entertainment Ali Wong’s stand up special delves into her sexual adventures, hoarding, the rocky road to pregnancy, and wh...",Ali Wong
anthony,Magazine of culture and entertainment There’s no subject too dark as the comedian skewers taboos and riffs on national tragedies before pulling ba...,Anthony Jeselnik
bill,"Magazine of culture and entertainment [cheers and applause] All right, thank you! Thank you very much! Thank you. Thank you. Thank you. How are yo...",Bill Burr
bo,Magazine of culture and entertainment Bo What? Old MacDonald had a farm E I E I O And on that farm he had a pig E I E I O Here a snort There a Old...,Bo Burnham
dave,Magazine of culture and entertainment The Age of Spin: Dave Chappelle Live at the Hollywood Palladium (2017) Dave Chappelle gives his usual skewed...,Dave Chappelle
hasan,"Magazine of culture and entertainment Comic Hasan Minhaj of “The Daily Show” shares personal stories about racism, immigrant parents, prom night h...",Hasan Minhaj
jim,"Magazine of culture and entertainment Nothing is sacred in this show from Australian comic Jim Jefferies, whether it’s the mother of his child, au...",Jim Jefferies
joe,,Joe Rogan
john,"Magazine of culture and entertainment Armed with boyish charm and a sharp wit, the former “SNL” writer John Mulaney offers sly takes on marriage, ...",John Mulaney
louis,"Magazine of culture and entertainment ‘Oh My God’ is the fifth comedy special performed by Louis C.K.. It premiered on HBO on April 13, 2013. Film...",Louis C.K.


In [21]:
# Let's pickle it for later use (overwrites pickle/corpus.pkl if it already exists)
import os
os.makedirs("pickle", exist_ok=True)
data_df.to_pickle("pickle/corpus.pkl")

### b) Document-Term Matrix

For many of the techniques we'll be using in future notebooks, the text must be tokenized, meaning broken down into smaller pieces. The most common tokenization technique is to break down text into words. We can do this using scikit-learn's CountVectorizer, where every row will represent a different document and every column will represent a different word.

In addition, with CountVectorizer, we can remove stop words. Stop words are common words that add no additional meaning to text such as 'a', 'the', etc.

In [22]:
# We are going to create a document-term matrix using CountVectorizer, and exclude common English stop words
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')
data_cv = cv.fit_transform(data_clean.transcript)
data_dtm = pd.DataFrame(data_cv.toarray(), columns=cv.get_feature_names_out())
data_dtm.index = data_clean.index
data_dtm

,aaaaah,aaaaahhhhhhh,aaaaauuugghhhhhh,aaaahhhhh,aaah,aah,abc,abcs,ability,abject,...,zee,zen,zeppelin,zero,zillion,zombie,zombies,zoning,zoo,éclair
ali,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0
anthony,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
bill,1,0,0,0,0,0,0,1,0,0,...,0,0,0,1,1,1,1,1,0,0
bo,0,1,1,1,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
dave,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
hasan,0,0,0,0,0,0,0,0,0,0,...,2,1,0,1,0,0,0,0,0,0
jim,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
joe,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
john,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
louis,0,0,0,0,0,3,0,0,0,0,...,0,0,0,2,0,0,0,0,0,0


In [23]:
# Let's pickle it for later use (overwrites pickle/dtm.pkl if it already exists)
data_dtm.to_pickle("pickle/dtm.pkl")

In [24]:
# Let's also pickle the cleaned data (before we put it in document-term matrix format) and the CountVectorizer object
# (overwrites pickle/data_clean.pkl and pickle/cv.pkl if they already exist)
data_clean.to_pickle('pickle/data_clean.pkl')
pickle.dump(cv, open("pickle/cv.pkl", "wb"))

## Q3: RegEx for Cleaning Text
---

Can you add an additional regular expression to the clean_text_round2 function to further clean the text?

In [25]:
def clean_text_round3(text):
    '''Remove leftover site boilerplate and collapse repeated whitespace.'''
    text = re.sub(r'magazine of culture and entertainment', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

round3 = lambda x: clean_text_round3(x)

In [26]:
data_clean['transcript'] = data_clean.transcript.apply(round3)
data_clean

,transcript,tokens,tokens_no_stop
ali,ali wongs stand up special delves into her sexual adventures hoarding the rocky road to pregnancy and why feminism is terrible ladies and gentleme...,"[magazine, of, culture, and, entertainment, ali, wongs, stand, up, special, delves, into, her, sexual, adventures, hoarding, the, rocky, road, to,...","[magazine, culture, entertainment, ali, wongs, stand, special, delves, sexual, adventures, hoarding, rocky, road, pregnancy, feminism, terrible, l..."
anthony,theres no subject too dark as the comedian skewers taboos and riffs on national tragedies before pulling back the curtain on his provocative style...,"[magazine, of, culture, and, entertainment, theres, no, subject, too, dark, as, the, comedian, skewers, taboos, and, riffs, on, national, tragedie...","[magazine, culture, entertainment, theres, subject, dark, comedian, skewers, taboos, riffs, national, tragedies, pulling, curtain, provocative, st..."
bill,all right thank you thank you very much thank you thank you thank you how are you whats going on thank you its a pleasure to be here in the greate...,"[magazine, of, culture, and, entertainment, all, right, thank, you, thank, you, very, much, thank, you, thank, you, thank, you, how, are, you, wha...","[magazine, culture, entertainment, right, thank, thank, thank, thank, thank, whats, going, thank, pleasure, greater, atlanta, georgia, area, oasis..."
bo,bo what old macdonald had a farm e i e i o and on that farm he had a pig e i e i o here a snort there a old macdonald had a farm e i e i o this is...,"[magazine, of, culture, and, entertainment, bo, what, old, macdonald, had, a, farm, e, i, e, i, o, and, on, that, farm, he, had, a, pig, e, i, e, ...","[magazine, culture, entertainment, bo, old, macdonald, farm, e, e, o, farm, pig, e, e, o, snort, old, macdonald, farm, e, e, o, bo, burnham, hes, ..."
dave,the age of spin dave chappelle live at the hollywood palladium dave chappelle gives his usual skewed insight into the topics of race technology oj...,"[magazine, of, culture, and, entertainment, the, age, of, spin, dave, chappelle, live, at, the, hollywood, palladium, dave, chappelle, gives, his,...","[magazine, culture, entertainment, age, spin, dave, chappelle, live, hollywood, palladium, dave, chappelle, gives, usual, skewed, insight, topics,..."
hasan,comic hasan minhaj of the daily show shares personal stories about racism immigrant parents prom night horrors and more in this standup special wh...,"[magazine, of, culture, and, entertainment, comic, hasan, minhaj, of, the, daily, show, shares, personal, stories, about, racism, immigrant, paren...","[magazine, culture, entertainment, comic, hasan, minhaj, daily, shares, personal, stories, racism, immigrant, parents, prom, night, horrors, stand..."
jim,nothing is sacred in this show from australian comic jim jefferies whether its the mother of his child auditioning disabled actors or gun control ...,"[magazine, of, culture, and, entertainment, nothing, is, sacred, in, this, show, from, australian, comic, jim, jefferies, whether, its, the, mothe...","[magazine, culture, entertainment, sacred, australian, comic, jim, jefferies, mother, child, auditioning, disabled, actors, gun, control, ladies, ..."
joe,,[],[]
john,armed with boyish charm and a sharp wit the former snl writer john mulaney offers sly takes on marriage his beef with babies and the time he met b...,"[magazine, of, culture, and, entertainment, armed, with, boyish, charm, and, a, sharp, wit, the, former, snl, writer, john, mulaney, offers, sly, ...","[magazine, culture, entertainment, armed, boyish, charm, sharp, wit, snl, writer, john, mulaney, offers, sly, takes, marriage, beef, babies, time,..."
louis,oh my god is the fifth comedy special performed by louis ck it premiered on hbo on april filmed in phoenix arizona at the celebrity theatre introf...,"[magazine, of, culture, and, entertainment, oh, my, god, is, the, fifth, comedy, special, performed, by, louis, ck, it, premie

## Q4: `CountVectorizer` parameters
---

Play around with `CountVectorizer`'s parameters. What is ngram_range? What is min_df and max_df?

In [27]:
# Tinkering with ngram_range: what changes if we let CountVectorizer count word pairs (bigrams) in addition to single words (unigrams)?
cv_unigram = CountVectorizer(stop_words='english')
cv_bigram = CountVectorizer(stop_words='english', ngram_range=(1, 2))

dtm_unigram = cv_unigram.fit_transform(data_clean.transcript)
dtm_bigram = cv_bigram.fit_transform(data_clean.transcript)

print('unigrams only, vocab size:', len(cv_unigram.get_feature_names_out()))
print('unigrams + bigrams, vocab size:', len(cv_bigram.get_feature_names_out()))
print('sample bigram features:', [f for f in cv_bigram.get_feature_names_out() if ' ' in f][:10])

unigrams only, vocab size: 7299
unigrams + bigrams, vocab size: 44004
sample bigram features: ['aaaaah anybody', 'aaaaahhhhhhh oh', 'aaaaauuugghhhhhh hard', 'aaaahhhhh aaaaahhhhhhh', 'aaah shit', 'aah aah', 'aah ones', 'abc yeah', 'abcs know', 'ability generate']


In [28]:
# Tinkering with min_df and max_df: how much does filtering by document frequency shrink the vocabulary compared to the default?
cv_default = CountVectorizer(stop_words='english')
cv_min_df = CountVectorizer(stop_words='english', min_df=2)     # drop words seen in <2 documents
cv_max_df = CountVectorizer(stop_words='english', max_df=0.5)   # drop words seen in >50% of documents

for name, vec in [('default', cv_default), ('min_df=2', cv_min_df), ('max_df=0.5', cv_max_df)]:
    vec.fit(data_clean.transcript)
    print(f'{name}: vocab size = {len(vec.get_feature_names_out())}')

default: vocab size = 7299
min_df=2: vocab size = 2697
max_df=0.5: vocab size = 6808


**What is `ngram_range`?**

`ngram_range=(min_n, max_n)` controls how many *consecutive* words `CountVectorizer` groups into one feature. The default `(1, 1)` only counts single words (unigrams). Above, setting it to `(1, 2)` keeps the unigrams and also adds every consecutive word pair (bigram) as its own feature, e.g. "not" and "funny" become three features: "not", "funny", and "not funny". That's why the unigram+bigram vocab printed above is always noticeably larger than the unigram-only one: it's roughly unigrams plus (almost) every adjacent word pair in the corpus. The upside is it can capture short phrases whose meaning depends on word order; the downside is a much larger, sparser matrix.

**What are `min_df` and `max_df`?**

Both filter the vocabulary by *document frequency*, i.e. how many transcripts a word appears in, NOT its raw count!
- `min_df` drops a word if it appears in **fewer** documents than this threshold (an int) or fraction of documents (a float). `min_df=2` above removes any word used by only one comedian — mostly names, one-off references, and typos that don't generalize across the corpus.
- `max_df` drops a word if it appears in **more** documents than this threshold/fraction. `max_df=0.5` above removes words present in over half the transcripts — words so common across every comedian's set that they stop helping distinguish one from another, functionally acting like extra, corpus-specific stop words.

Both `min_df` and `max_df` runs above should come out with a *smaller* vocab than the default, as `min_df` prunes the noisy long tail, and `max_df` prunes the over-common middle. Combining both is a quick way to tighten the vocabulary before feeding it into a model.